In [ ]:
import os
import json

import dask.dataframe as dd
import geopandas as gpd
import numpy as np
import pandas as pd

## Functions

In [ ]:
def getFiles(filepath):
    files = []
    for file in os.listdir(filepath):
        if not file.startswith('.'):
            # if not 'C2' in file:
            files.append(os.path.join(filepath, file))
    return files

## Setup

In [ ]:
data_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/data/'
output_dir = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_output/PIXC_vD_wse_2026_06_01_rel_abs/'
save_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results/'

# Order of all possible bins for plotting
order = ['(0, 10]', '(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]', '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 500]', '(500, 1000]',
         '(1000, 5000]']

# Cutpoints
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 150, 200, 500, 1000, 5000]

## Harlan et al. 2026

In [ ]:
df_sp = pd.read_csv(os.path.join(data_dir, 'paired_node_SWOT_gage.csv'),
                    dtype={'gage_id': str, 'sword_version':str})

In [ ]:
df_sp['node_q_b'].describe()

In [ ]:
# Check that our max bin width is correct
assert df_sp.p_width.max() <= bins[-1]

In [ ]:
# Bin reaches by width for pre-filtering analysis
df_sp.loc[:, 'Bin'] = pd.cut(df_sp['p_width'], bins).astype(str)
df_sp['Bin'] = pd.Categorical(df_sp['Bin'], categories=order, ordered=True)
df_sp = df_sp.reset_index()
df_sp = df_sp.drop(columns='index')

In [ ]:
# Count number of reaches (at gauges), obs by width bin before filtering
results = {}

results['unfiltered'] = df_sp.groupby('Bin', observed=False)['gage_id'].agg(
    reaches='nunique', 
    obs='count'
).reset_index()

# Create columns needed for quality filtering
df_sp.loc[:, 'obs_area_frac'] = df_sp['area_detct'] / df_sp['area_total']
df_sp.loc[:, 'wse_pdiff'] = np.abs(df_sp['wse'] - df_sp['p_wse'])

# Sort data
df_sp = df_sp.sort_values(['gage_id', 'gage_time_UTC'])

In [ ]:
# Mask following Harlan et al. 2026 EXCEPT for n_good_pix
mask = (
    (df_sp.SWOT_data_version == 'D') &
    (df_sp.node_q < 3) &
    # Partial coverage
    (df_sp.ice_clim_f == 0) &
    (df_sp.xovr_cal_q <= 1) &
    # Hampel filter below
    (df_sp.node_q_b < 4194304) &
    (df_sp.dark_frac < 0.5) &
    (df_sp.xtrk_dist.abs().between(10000, 60000)) &
    (df_sp.wse > -998) &
    (df_sp.obs_area_frac >= 0.5) & # df_sp['area_detct'] / df_sp['area_total']
    (df_sp.wse_pdiff < 10) & # np.abs(df_sp['wse'] - df_sp['p_wse'])
    (df_sp.n_good_pix >= 5)
)

In [ ]:
# # Apply Hampel filtering--NO LONGER APPLYING
# df_sp['is_outlier'] = df_sp.groupby(['gage_id', 'node_id'])['wse'].transform(
#     lambda x: hampel_filter(series=x, window_size=3)
# )
# df_sp = df_sp[~df_sp['is_outlier']]

# Apply quality mask
df_sp = df_sp.loc[mask]

# Calculate various error terms
groups = df_sp.groupby(['gage_id', 'node_id'])

df_sp = df_sp.assign(
    wse_gauge_norm = df_sp['gage_stage_m'] - groups['gage_stage_m'].transform('median'),
    wse_swot_norm  = df_sp['wse'] - groups['wse'].transform('median')
)

df_sp['wse_bias'] = df_sp['wse'] - df_sp['gage_stage_m']
df_sp['wse_diff'] = df_sp['wse_swot_norm'] - df_sp['wse_gauge_norm']
df_sp['wse_diff_abs'] = df_sp['wse_diff'].abs()

# Final filtering
df_sp = df_sp[df_sp['wse_diff_abs'] < 5]
df_sp = df_sp.groupby('gage_id').filter(lambda x: len(x) >= 20)

print(len(df_sp.gage_id.unique()))

df_sp = df_sp.reset_index()

# Write out Harlan et al. 2026 results
df_sp.to_parquet(os.path.join(save_dir, 'harlan_wse_observations.parquet'))

In [ ]:
# Count number of reaches (at gauges), obs by width bin after filtering
results['filtered'] = df_sp.groupby('Bin', observed=False)['gage_id'].agg(
    reaches='nunique', 
    obs='count'
).reset_index()

## My data

In [ ]:
# Get list of files
files = getFiles(output_dir)
print(len(files))

# Read in all files
ddf = dd.read_parquet(files)

# Bin reaches by width for pre-filtering analysis
ddf['Bin'] = ddf['WidthM'].map_partitions(pd.cut, bins=bins).astype(str)
ddf = ddf.reset_index(drop=True)

In [ ]:
# Count number of reaches before filtering
unfiltered_task = ddf.groupby('Bin', observed=False)['site_id'].nunique()
unfiltered = unfiltered_task.compute().reset_index(name='count_unfil')

unfiltered['Bin'] = pd.Categorical(unfiltered['Bin'], categories=order, ordered=True)
unfiltered = unfiltered.sort_values('Bin')

max_w = ddf.WidthM.max().compute()

In [ ]:
# Sort data
ddf = ddf[
    ddf['wse_gauge'].notnull() & 
    ddf['wse_swot'].notnull()
    ][[
        'date', 'n_pixels', 'wse_gauge',
        'wse_swot', 'site_id', 'WidthM',
        'Bin'
    ]]

df = ddf.compute().sort_values(['site_id', 'date'])

print(len(df.site_id.unique()))

In [ ]:
# Write out site_ids for gauges with any data after pre-processsing
any_data_ids = list(df.site_id.unique())

with open(os.path.join(data_dir, 'any_data_post_ids.json'), 'w') as f:
    json.dump(any_data_ids, f, indent=4)

In [ ]:
# Apply quality mask
df = df[df['n_pixels'] >= 5].copy()
df = df.groupby('site_id').filter(lambda x: len(x) >= 20)

# Calculate various error terms
groups = df.groupby('site_id')

df = df.assign(
    wse_gauge_norm = df['wse_gauge'] - groups['wse_gauge'].transform('median'),
    wse_swot_norm  = df['wse_swot'] - groups['wse_swot'].transform('median')
)

df['wse_bias'] = df['wse_swot'] - df['wse_gauge']
df['wse_diff'] = df['wse_swot_norm'] - df['wse_gauge_norm']
df['wse_diff_abs'] = df['wse_diff'].abs()

# Final filtering
df = df[df['wse_diff_abs'] < 5].copy()

df.to_parquet(os.path.join(save_dir, 'bennitt_wse_observations.parquet'))

In [ ]:
useable = df.groupby('Bin', observed=False)['site_id'].nunique().reset_index(name='count_useable')

print(useable.count_useable.sum())

diff = df.groupby(['Bin', 'site_id'], observed=False)['wse_diff_abs'].median()
diff = diff.reset_index()
diff.columns = ['Bin', 'site_id', 'median_wse_diff_abs']

heights = diff[diff['median_wse_diff_abs'] < 0.3].groupby('Bin', observed=False)['site_id'].nunique().reset_index(name='count_heights')

corr = df.groupby(['Bin', 'site_id'], observed=False)[['wse_gauge_norm', 'wse_swot_norm']].corr()
corr = corr.unstack().iloc[:, 1].reset_index()
corr.columns = ['Bin', 'site_id', 'correlation']

stats = corr.merge(diff, on=['site_id', 'Bin'])
high_qual = stats[(stats['correlation'] > 0.6) & (np.abs(stats['median_wse_diff_abs']) < 0.3)]
high_qual_ids = high_qual.site_id.to_list()
high_qual = high_qual.groupby('Bin', observed=False)['site_id'].nunique().reset_index(name='count_high_qual')

In [ ]:
print(stats[stats['Bin'].isin(order[5:])]['correlation'].median())

print(stats[stats['Bin'].isin(order[5:])]['median_wse_diff_abs'].median())

In [ ]:
with open(os.path.join(data_dir, 'high_qual_ids.json') 'w') as f:
    json.dump(high_qual_ids, f, indent=4)

df_qual = unfiltered.merge(useable, on='Bin')
df_qual = df_qual.merge(heights, on='Bin')
df_qual = df_qual.merge(high_qual, on='Bin')

df_qual['Bin'] = pd.Categorical(df_qual['Bin'], categories=order, ordered=True)
df_qual = df_qual.sort_values('Bin')

df_qual.to_parquet(os.path.join(save_dir, 'bennitt_wse_quality.parquet'))

In [ ]:
temp = df_qual[df_qual['Bin'].isin(order[5:])].set_index('Bin').sum()

print(temp)
print(temp['count_high_qual'] / temp['count_unfil'])
print(temp['count_heights'] / temp['count_unfil'])